In [ ]:
!pip install transformers datasets sacrebleu sentencepiece sacremoses -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 25.3 MB/s eta 0:00:00


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving combined_final.csv to combined_final.csv


In [ ]:
import pandas as pd
import random

df = pd.read_csv("combined_final.csv")

# --- Basic stats ---
print(f"Total pairs: {len(df)}")
print(f"Unique roman spellings: {df['roman'].nunique()}")
print(f"Unique Khmer outputs: {df['khmer'].nunique()}")
print()

# --- Length stats ---
df['roman_len'] = df['roman'].str.len()
df['khmer_len'] = df['khmer'].str.len()
print(f"Avg roman length: {df['roman_len'].mean():.1f} chars")
print(f"Avg Khmer length: {df['khmer_len'].mean():.1f} chars")
print(f"Max roman length: {df['roman_len'].max()} chars")
print()

# --- Sample rows ---
print("=== SAMPLE PAIRS ===")
print(df.sample(10).to_string(index=False))
print()

# --- Important insight ---
# Notice that many roman spellings map to the SAME Khmer output.
# Example: 'tov', 'Tov', 'tv' all → ទៅ
# This is the "spelling variation" problem we're solving!
print("=== AMBIGUITY EXAMPLE ===")
print("Words meaning ទៅ (go):")
print(df[df['khmer'] == 'ទៅ']['roman'].tolist())

Total pairs: 1162
Unique roman spellings: 1162
Unique Khmer outputs: 681

Avg roman length: 7.0 chars
Avg Khmer length: 6.0 chars
Max roman length: 41 chars

=== SAMPLE PAIRS ===
           roman      khmer  roman_len  khmer_len
             hev        ហេវ          3          3
            kert        កើត          4          3
          thlong      ថ្លង់          6          5
           t'lon    ទ្រលាន់          5          7
          tortul       ទទួល          6          4
             sur        សួរ          3          3
chngay cheang ke ឆ្ងាយជាងគេ         16         10
           p'lom      បន្លំ          5          5
         tuk tuk     តុកតុក          7          6
       ton samai   ទាន់សម័យ          9          8

=== AMBIGUITY EXAMPLE ===
Words meaning ទៅ (go):
['tv', 'tov']


In [ ]:
import re

augmented = df[['roman', 'khmer']].copy()
new_rows = []

for _, row in df.iterrows():
    r = row['roman'].strip()
    k = row['khmer'].strip()

    # Variant 1: All lowercase
    new_rows.append({'roman': r.lower(), 'khmer': k})

    # Variant 2: First letter capitalized
    new_rows.append({'roman': r.capitalize(), 'khmer': k})

    # Variant 3: Remove apostrophes (e.g. "k'mean" → "kmean")
    no_apos = r.replace("'", "").lower()
    if no_apos != r.lower():
        new_rows.append({'roman': no_apos, 'khmer': k})

    # Variant 4: Replace common alternate spellings
    # "ey" ↔ "ei", "ou" ↔ "o", "ae" ↔ "e"
    for old, new in [("ey", "ei"), ("ou", "o"), ("ae", "e"), ("oa", "o"), ("ea", "ia")]:
        alt = re.sub(old, new, r.lower())
        if alt != r.lower():
            new_rows.append({'roman': alt, 'khmer': k})

aug_df = pd.DataFrame(new_rows)
full_df = pd.concat([augmented, aug_df], ignore_index=True)
full_df = full_df.drop_duplicates(subset=['roman']).dropna()

print(f"Before augmentation: {len(df)} pairs")
print(f"After augmentation:  {len(full_df)} pairs")
print(f"Growth: +{len(full_df) - len(df)} new training examples")

NameError: name 'df' is not defined

In [ ]:
def to_char_level(text):
    """Convert 'slanh' → 's l a n h'"""
    return " ".join(list(str(text).strip()))

# Apply to full dataset
full_df['roman_chars'] = full_df['roman'].apply(to_char_level)
full_df['khmer_chars'] = full_df['khmer'].apply(to_char_level)

# Show the transformation
print("=== BEFORE vs AFTER char-level encoding ===")
for _, row in full_df.sample(5).iterrows():
    print(f"  Input:  '{row['roman']}' → '{row['roman_chars']}'")
    print(f"  Output: '{row['khmer']}' → '{row['khmer_chars']}'")
    print()

=== BEFORE vs AFTER char-level encoding ===
  Input:  'arkoun' → 'a r k o u n'
  Output: 'អរគុណ' → 'អ រ គ ុ ណ'

  Input:  'pit brakot' → 'p i t   b r a k o t'
  Output: 'ពិតប្រាកដ' → 'ព ិ ត ប ្ រ ា ក ដ'

  Input:  'jivit' → 'j i v i t'
  Output: 'ជីវិត' → 'ជ ី វ ិ ត'

  Input:  'kok kdav' → 'k o k   k d a v'
  Output: 'កក់ក្ដៅ' → 'ក ក ់ ក ្ ដ ៅ'

  Input:  'kmean ros chiet' → 'k m e a n   r o s   c h i e t'
  Output: 'គ្មានរស់ជាតិគ្មាន' → 'គ ្ ម ា ន រ ស ់ ជ ា ត ិ គ ្ ម ា ន'



In [ ]:
from sklearn.model_selection import train_test_split

# First split off 20% for val+test
train_df, temp_df = train_test_split(full_df, test_size=0.2, random_state=42)

# Split that 20% evenly into val and test
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f"Train: {len(train_df)} pairs ({len(train_df)/len(full_df)*100:.0f}%)")
print(f"Val:   {len(val_df)} pairs ({len(val_df)/len(full_df)*100:.0f}%)")
print(f"Test:  {len(test_df)} pairs ({len(test_df)/len(full_df)*100:.0f}%)")

# Save splits for later use
train_df.to_csv("train.csv", index=False)
val_df.to_csv("val.csv", index=False)
test_df.to_csv("test.csv", index=False)
print("\nSaved train.csv, val.csv, test.csv")

Train: 2204 pairs (80%)
Val:   275 pairs (10%)
Test:  276 pairs (10%)

Saved train.csv, val.csv, test.csv


In [ ]:
from transformers import MarianMTModel, MarianTokenizer

# We use a small English→Romance model as our starting point.
# It's not Khmer-specific, but the architecture is what matters.
# The Transformer layers understand "sequence patterns" regardless of language.
MODEL_NAME = "Helsinki-NLP/opus-mt-en-ROMANCE"

print("Loading tokenizer...")
tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)

print("Loading model...")
model = MarianMTModel.from_pretrained(MODEL_NAME)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nModel loaded!")
print(f"Total parameters: {total_params:,}")
print(f"That's {total_params/1e6:.1f} million numbers the model will adjust during training.")

Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/265 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/779k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/799k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]


Model loaded!
Total parameters: 144,504,320
That's 144.5 million numbers the model will adjust during training.


In [ ]:
from datasets import Dataset
import torch

MAX_INPUT_LEN = 80   # max characters in romanized input (× 2 because char-level)
MAX_TARGET_LEN = 80  # max characters in Khmer output

def tokenize_batch(batch):
    model_inputs = tokenizer(
        batch["roman_chars"],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=batch["khmer_chars"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Convert pandas DataFrames to HuggingFace Dataset objects
train_dataset = Dataset.from_pandas(train_df[['roman_chars', 'khmer_chars']])
val_dataset   = Dataset.from_pandas(val_df[['roman_chars', 'khmer_chars']])
test_dataset  = Dataset.from_pandas(test_df[['roman_chars', 'khmer_chars']])

# Apply tokenization (batched = process 32 rows at a time, faster)
print("Tokenizing train set...")
train_tok = train_dataset.map(tokenize_batch, batched=True, batch_size=32)

print("Tokenizing val set...")
val_tok = val_dataset.map(tokenize_batch, batched=True, batch_size=32)

print("Tokenizing test set...")
test_tok = test_dataset.map(tokenize_batch, batched=True, batch_size=32)

print(f"\nDone! Train tokens shape: {len(train_tok)} examples")
print("Each example contains: input_ids, attention_mask, labels")

Tokenizing train set...


Map:   0%|          | 0/2204 [00:00<?, ? examples/s]

Tokenizing val set...


Map:   0%|          | 0/275 [00:00<?, ? examples/s]

Tokenizing test set...


Map:   0%|          | 0/276 [00:00<?, ? examples/s]


Done! Train tokens shape: 2204 examples
Each example contains: input_ids, attention_mask, labels


In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq, EarlyStoppingCallback

training_args = Seq2SeqTrainingArguments(
    output_dir="./synctype-checkpoints",
    num_train_epochs=50,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=3e-5,
    warmup_steps=100,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    fp16=True,
    logging_steps=20,
    report_to="none",
)
print("Config ready")




Config ready


In [ ]:
import numpy as np
from sacrebleu.metrics import CHRF

chrf_metric = CHRF()

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # Decode predicted token IDs back to text
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    # Replace -100 (padding label) with pad token ID, then decode
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Remove spaces (we added them for char-level encoding)
    decoded_preds  = [p.replace(" ", "") for p in decoded_preds]
    decoded_labels = [l.replace(" ", "") for l in decoded_labels]

    # Character accuracy: exact match
    exact_matches = sum(p == r for p, r in zip(decoded_preds, decoded_labels))
    char_acc = exact_matches / len(decoded_labels)

    # chrF score
    chrf_score = chrf_metric.corpus_score(decoded_preds, [decoded_labels]).score

    return {
        "char_accuracy": round(char_acc, 4),
        "chrf": round(chrf_score, 2)
    }

print("Metrics defined: char_accuracy + chrF")
print("char_accuracy = exact word match rate")
print("chrF = partial credit for near-correct outputs")import numpy as np
from sacrebleu.metrics import CHRF

chrf_metric = CHRF()

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # Decode predicted token IDs back to text
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    # Replace -100 (padding label) with pad token ID, then decode
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Remove spaces (we added them for char-level encoding)
    decoded_preds  = [p.replace(" ", "") for p in decoded_preds]
    decoded_labels = [l.replace(" ", "") for l in decoded_labels]

    # Character accuracy: exact match
    exact_matches = sum(p == r for p, r in zip(decoded_preds, decoded_labels))
    char_acc = exact_matches / len(decoded_labels)

    # chrF score
    chrf_score = chrf_metric.corpus_score(decoded_preds, [decoded_labels]).score

    return {
        "char_accuracy": round(char_acc, 4),
        "chrf": round(chrf_score, 2)
    }

print("Metrics defined: char_accuracy + chrF")
print("char_accuracy = exact word match rate")
print("chrF = partial credit for near-correct outputs")

Metrics defined: char_accuracy + chrF
char_accuracy = exact word match rate
chrF = partial credit for near-correct outputs


In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer, model=model,
    padding=True,
    pad_to_multiple_of=8
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]  # stops if no improvement for 5 epochs
)

print("=" * 50)
print("Starting training!")
print("=" * 50)

trainer.train()

trainer.save_model("./synctype-final")
tokenizer.save_pretrained("./synctype-final")
print("\nModel saved!")

NameError: name 'DataCollatorForSeq2Seq' is not defined

In [ ]:
from transformers import MarianMTModel, MarianTokenizer
import torch

# Load saved model
model = MarianMTModel.from_pretrained("./synctype-final")
tokenizer = MarianTokenizer.from_pretrained("./synctype-final")
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Load the lookup dictionary (for the hybrid system)
lookup = {}
df_full = pd.read_csv("combined_final.csv")
for _, row in df_full.iterrows():
    lookup[str(row['roman']).strip().lower()] = str(row['khmer']).strip()

def predict_single(roman_word):
    """Predict Khmer script for one romanized word."""
    chars = " ".join(list(roman_word.lower().strip()))
    inputs = tokenizer([chars], return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        output = model.generate(**inputs, num_beams=4, max_length=80)
    decoded = tokenizer.decode(output[0], skip_special_tokens=True)
    return decoded.replace(" ", "")  # remove char-level spaces

def hybrid_predict(text):
    """
    HYBRID SYSTEM: lookup dict first (fast, exact match),
    fall back to model for unknown words.
    This is how production systems like Google Pinyin work.
    """
    words = text.lower().strip().split()
    results = []
    for w in words:
        if w in lookup:
            results.append(("dict", lookup[w]))
        else:
            results.append(("model", predict_single(w)))
    return "".join(r[1] for r in results), results

# --- Run evaluation ---
print("Running evaluation on test set...")
preds, refs = [], []
errors = []

for _, row in test_df.iterrows():
    pred = predict_single(row['roman'])
    ref  = row['khmer'].strip()
    preds.append(pred)
    refs.append(ref)
    if pred != ref:
        errors.append((row['roman'], pred, ref))

# Calculate metrics
exact_matches = sum(p == r for p, r in zip(preds, refs))
char_accuracy = exact_matches / len(refs)
chrf_score = chrf_metric.corpus_score(preds, [refs]).score

print("\n" + "=" * 50)
print("FINAL EVALUATION RESULTS")
print("=" * 50)
print(f"Test examples: {len(refs)}")
print(f"Exact match accuracy: {char_accuracy*100:.1f}%")
print(f"chrF score: {chrf_score:.1f} / 100")
print()

# Show some errors — very informative!
print(f"ERRORS ({len(errors)} / {len(refs)}):")
for roman, pred, ref in errors[:10]:
    print(f"  '{roman}' → predicted: '{pred}' | correct: '{ref}'")

Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Running evaluation on test set...

FINAL EVALUATION RESULTS
Test examples: 276
Exact match accuracy: 76.8%
chrF score: 90.8 / 100

ERRORS (64 / 276):
  'Mer' → predicted: 'មែរ' | correct: 'មើល'
  'suy' → predicted: 'សួយ' | correct: 'ស្អុយ'
  'Hous' → predicted: 'ហោះ' | correct: 'ហួស'
  'S'kom' → predicted: 'សាកម' | correct: 'ស្គម'
  'osa' → predicted: 'អស្សា' | correct: 'ឧស្សា'
  'Krong pailin' → predicted: 'គ្រុងប៉លិន' | correct: 'ក្រុងប៉ៃលិន'
  'Tngai lech' → predicted: 'ថ្ងល្ច' | correct: 'ថ្ងៃល្ច'
  'Neng' → predicted: 'នឹង' | correct: 'និង'
  'hos' → predicted: 'ហុះ' | correct: 'ហួស'
  'Mech' → predicted: 'មេច' | correct: 'ម៉េច'


In [ ]:
print("=" * 50)
print("BASELINE vs MODEL COMPARISON")
print("=" * 50)

baseline_preds = []
for _, row in test_df.iterrows():
    r = row['roman'].strip().lower()
    if r in lookup:
        baseline_preds.append(lookup[r])
    else:
        baseline_preds.append("")  # unknown

baseline_exact = sum(p == r for p, r in zip(baseline_preds, refs))
baseline_acc = baseline_exact / len(refs)
baseline_chrf = chrf_metric.corpus_score(baseline_preds, [refs]).score

print(f"{'Method':<20} {'Exact Acc':>10} {'chrF':>8}")
print("-" * 40)
print(f"{'Baseline (dict)':<20} {baseline_acc*100:>9.1f}% {baseline_chrf:>7.1f}")
print(f"{'MarianMT Model':<20} {char_accuracy*100:>9.1f}% {chrf_score:>7.1f}")
print()
print("The model should score HIGHER than baseline on unseen words.")
print("If not: add more training data and re-train.")

BASELINE vs MODEL COMPARISON
Method                Exact Acc     chrF
----------------------------------------
Baseline (dict)           83.3%    83.6
MarianMT Model            76.8%    90.8

The model should score HIGHER than baseline on unseen words.
If not: add more training data and re-train.


In [ ]:
print("=" * 50)
print("LIVE DEMO — Hybrid Transliteration")
print("=" * 50)

test_sentences = [
    "slanh eng nas",
    "besdong tver ey",
    "yg chea khmer",
    "jg rorm",
    "mean terk jit",
    "krous tnak nas",
    "reakreay klang",
    "akk jeang ke",
]

for sentence in test_sentences:
    output, details = hybrid_predict(sentence)
    sources = [f"{w}({'📖' if s=='dict' else '🤖'})" for s, w in details]
    print(f"  '{sentence}'")
    print(f"  → {output}")
    print(f"  Sources: {' '.join(sources)}")
    print()

LIVE DEMO — Hybrid Transliteration
  'slanh eng nas'
  → ស្រឡាញ់ឯងណាស់
  Sources: ស្រឡាញ់(📖) ឯង(📖) ណាស់(📖)

  'besdong tver ey'
  → បេះដូងទ្វើរអី
  Sources: បេះដូង(📖) ទ្វើរ(🤖) អី(📖)

  'yg chea khmer'
  → យើងជាខ្មែរ
  Sources: យើង(📖) ជា(📖) ខ្មែរ(📖)

  'jg rorm'
  → ចង់រាំ
  Sources: ចង់(📖) រាំ(🤖)

  'mean terk jit'
  → មានទឹកជិត
  Sources: មាន(📖) ទឹក(📖) ជិត(📖)

  'krous tnak nas'
  → គ្រស់ថ្នាក់ណាស់
  Sources: គ្រស់(🤖) ថ្នាក់(🤖) ណាស់(📖)

  'reakreay klang'
  → រីករាយខ្លាំង
  Sources: រីករាយ(📖) ខ្លាំង(📖)

  'akk jeang ke'
  → អាក្រក់ជាងគេ
  Sources: អាក្រក់(📖) ជាង(🤖) គេ(📖)



In [ ]:
import shutil

# Zip the model directory
shutil.make_archive("synctype_model", 'zip', "./synctype-final")

# Download the zip
from google.colab import files
files.download("synctype_model.zip")

# Also save test results
results_df = pd.DataFrame({
    "roman": test_df['roman'].values,
    "predicted": preds,
    "correct": refs,
    "exact_match": [p == r for p, r in zip(preds, refs)]
})
results_df.to_csv("test_results.csv", index=False)
files.download("test_results.csv")

print("Downloaded! Keep synctype_model.zip for the Gradio demo.")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded! Keep synctype_model.zip for the Gradio demo.
